# Predictive analysis — classification
Train/test split, cross-validated hyperparameter tuning and independent test evaluation. Models: Logistic Regression, SVM, Decision Tree, KNN. Avoid data leakage: preprocessing is fitted within each training fold.

In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,classification_report
D=Path('../data');O=Path('../outputs');O.mkdir(exist_ok=True)
df=pd.read_csv(D/'dataset_part_2.csv')
features=[c for c in ['FlightNumber','PayloadMass','Orbit','LaunchSite','Flights','GridFins','Reused','Legs'] if c in df.columns]
X=df[features].copy();y=df['Class'].astype(int)
num=X.select_dtypes(include=['number','bool']).columns.tolist();cat=[c for c in features if c not in num]
print('Features:',features,'Classes:',y.value_counts().to_dict())
if y.nunique()<2: raise ValueError('Need both landing classes')
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])

In [ ]:
models={
 'Logistic Regression':(LogisticRegression(max_iter=2000),{'model__C':[.01,.1,1,10]}),
 'SVM':(SVC(),{'model__C':[.1,1,10],'model__kernel':['linear','rbf']}),
 'Decision Tree':(DecisionTreeClassifier(random_state=42),{'model__max_depth':[2,4,6,None],'model__min_samples_leaf':[1,3,5]}),
 'KNN':(KNeighborsClassifier(),{'model__n_neighbors':[3,5,7,9]})}
minclass=y_train.value_counts().min();folds=min(5,int(minclass))
if folds<2: raise ValueError('Too few examples of one class for cross-validation')
cv=StratifiedKFold(n_splits=folds,shuffle=True,random_state=42)
results=[];fitted={}
for name,(est,grid) in models.items():
    search=GridSearchCV(Pipeline([('pre',pre),('model',est)]),grid,cv=cv,scoring='accuracy',n_jobs=-1)
    search.fit(X_train,y_train);pred=search.predict(X_test)
    results.append({'Model':name,'CV_accuracy':search.best_score_,'Test_accuracy':accuracy_score(y_test,pred),'Best_params':str(search.best_params_)})
    fitted[name]=search
scores=pd.DataFrame(results).sort_values('Test_accuracy',ascending=False)
display(scores);scores.to_csv(O/'model_scores.csv',index=False)

In [ ]:
ax=scores.plot.bar(x='Model',y='Test_accuracy',legend=False,figsize=(9,4),ylim=(0,1),title='Held-out test accuracy')
plt.ylabel('Accuracy');plt.tight_layout();plt.savefig(O/'model_accuracy.png',dpi=180);plt.show()
# Select by CV performance, not by test performance, to avoid test-set model selection bias.
best_name=max(fitted,key=lambda k:fitted[k].best_score_)
best=fitted[best_name];pred=best.predict(X_test)
print('Model selected by CV:',best_name);print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues');plt.title(best_name);plt.tight_layout();plt.savefig(O/'confusion_matrix.png',dpi=180);plt.show()